In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# for better resolution plots
%config InlineBackend.figure_format = 'retina'

sns.set_theme()

# Is it worth using
# We need to analyse it and decide if merging it is an option
clientes = pd.read_csv("data/clientes.csv", index_col="customer_id",
                       float_precision="round_trip")

df = pd.read_csv("data/train.csv", index_col="ID",
                       float_precision="round_trip")

Database dimensions:


(6534, 38)

In [1]:
print("Target variable value counts:")
print(df["repeat_purchase_90d"].value_counts())

print("\nTarget variable percentages:")
print((df["repeat_purchase_90d"].value_counts(normalize=True) * 100).round(2))

Target variable value counts:


NameError: name 'df' is not defined

We can see that the database has a decent size (6534 rows) and the target value is decently well distributed.
Our goal is to predict if clients are going to repeat a purchase with the highest hit rate possible.

We have 2 id's the first (ID) iterates all the train database, the second (customer_id) is our foreign key that connects the clients database.

In [5]:
pd.set_option('display.max_columns', None)
df.head(3)

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
10000000,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,1,79.0,126.0,2.0,858.30,429.150000,112.656252,349.49,508.81,47.000000,NaN,0.0,0.00,1.0,349.49,2.0,858.30,2.0,858.30,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,-0.660448
10000001,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,1,244.0,616.0,5.0,6722.83,1344.566000,1683.795178,84.38,4222.15,93.000000,117.657696,0.0,0.00,0.0,0.00,0.0,0.00,4.0,2500.68,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,0.924510
10000002,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,1,9.0,610.0,90.0,17590.08,195.445333,190.389932,4.54,1002.15,6.752809,6.384073,2.0,111.27,9.0,1928.55,28.0,5290.25,49.0,7899.80,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,-0.258730


In [6]:
clientes.head(3)

,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
customer_id,,,,,,,,,,,,
842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,City CY417: Missing Cell Port,NaN,Postal code PC828: #REF! Delivery Route,NaN,Address AD1120: Cell Without a Home,NaN,Excel Is Not A Database Ltd 150,2018
770941,Country CT9: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,Postal code PC56: #REF! Delivery Route,NaN,Address AD62: Cell Without a Home,NaN,Excel Is Not A Database Ltd 1064,2021
827053,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC827: #REF! Delivery Route,Street ST170: PowerPoint New Walk,Address AD449: Cell Without a Home,NaN,Excel Is Not A Database Ltd 113,2016


In [7]:
df.info()

<class 'pandas.DataFrame'>
Index: 6534 entries, 10000000 to 10007674
Data columns (total 38 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Total a Pagar                   6469 non-null   float64
 1   Total Bruto                     6469 non-null   float64
 2   Total Liquido                   6469 non-null   float64
 3   Total Imp_                      6469 non-null   float64
 4   Total Portes                    6469 non-null   float64
 5   Total Desc_ Global              6469 non-null   float64
 6   Total Desc_ Linha               6469 non-null   float64
 7   Custo                           6469 non-null   float64
 8   N_ Detalhes                     6469 non-null   float64
 9   invoice_count_current_purchase  6469 non-null   float64
 10  repeat_purchase_90d             6534 non-null   int64  
 11  days_since_previous_purchase    5673 non-null   float64
 12  customer_tenure_days            6469 no

In [8]:
df.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
Total a Pagar,6469.0,925.56,1335.00,0.93,188.92,476.74,1087.67,2.229721e+04
Total Bruto,6469.0,886.15,1327.64,0.83,175.74,445.74,1036.70,2.157828e+04
Total Liquido,6469.0,773.34,1136.36,0.76,154.38,389.91,903.37,1.812781e+04
Total Imp_,6469.0,152.22,225.26,0.00,29.93,79.74,184.84,4.169400e+03
Total Portes,6469.0,1.15,5.84,0.00,0.00,0.00,0.00,1.810600e+02
Total Desc_ Global,6469.0,0.35,5.12,0.00,0.00,0.00,0.00,2.184900e+02
Total Desc_ Linha,6469.0,112.47,289.99,0.00,0.00,13.13,92.28,4.103440e+03
Custo,6469.0,328.17,519.51,0.00,49.05,152.54,396.06,7.427400e+03
N_ Detalhes,6469.0,21.52,27.71,1.00,4.00,11.00,29.00,4.230000e+02
invoice_count_current_purchase,6469.0,1.07,0.31,1.00,1.00,1.00,1.00,5.000000e+00


Numerical variables with problems:
N_Detalhes, invoice_count_current_purchase, days_since_previous_purchase, customer_tenure_days, prior_purchase_count, purchase_count_30d, purchase_count_90d, purchase_count_180d, purchase_count_365d should be an int??
Many variables with negative values should be treated! (days_since_previous_purchase, prior_mean_purchase_value,  prior_mean_gap_days, spend_365d)

In [9]:
print(df.describe(include=['str']))

for col in df.select_dtypes(include=['str']).columns:
    print(f"\n{col}:")

    counts = df[col].value_counts(dropna=False)
    percentages = (df[col].value_counts(normalize=True, dropna=False) * 100).round(2)

    print(pd.DataFrame({
        "Count": counts,
        "Percentage (%)": percentages
    }))

                    payment_type       salesperson  \
count                       6274              6275   
unique                         9                 9   
top     Maybe Tomorrow Payment 7  VLOOKUP Master 1   
freq                        4565              4054   

                    commercial_zone            transport_method  \
count                          6469                        6273   
unique                            5                          80   
top     Decorative Dashboard Zone 4  Maybe Tomorrow Delivery 72   
freq                           6260                        1119   

                      warehouse  document_series purchase_date  
count                      6274             6469          6469  
unique                        2                3          1492  
top     Final Excel Warehouse 1  Ctrl Z Series 2    2015-10-31  
freq                       6253             2738            52  

payment_type:
                           Count  Percentage (%)
paym

How should we treat this variables??
Should we do Ordinal our One-hot encoding?
O transport_method devemos poder usar Categorical Encoding

In [31]:
# Duplicates
display(df[df.index.duplicated(keep=False)])

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


In [29]:
# Ver todas as linhas com ID repetido
display(df[df.index.duplicated(keep=False)])

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


In [11]:
# Missing Values
df.isna().sum()

Total a Pagar                       65
Total Bruto                         65
Total Liquido                       65
Total Imp_                          65
Total Portes                        65
Total Desc_ Global                  65
Total Desc_ Linha                   65
Custo                               65
N_ Detalhes                         65
invoice_count_current_purchase      65
repeat_purchase_90d                  0
days_since_previous_purchase       861
customer_tenure_days                65
prior_purchase_count                65
prior_total_spend                   65
prior_mean_purchase_value          862
prior_std_purchase_value          1113
prior_min_purchase_value           668
prior_max_purchase_value           668
prior_mean_gap_days               1308
prior_std_gap_days                1478
purchase_count_30d                  65
spend_30d                           65
purchase_count_90d                  65
spend_90d                           65
purchase_count_180d      

In [12]:
null_counts = df.isnull().sum()

cols_65 = null_counts[null_counts == 65].index.tolist()

print("Number of columns:", len(cols_65))
print(cols_65)

Number of columns: 24
['Total a Pagar', 'Total Bruto', 'Total Liquido', 'Total Imp_', 'Total Portes', 'Total Desc_ Global', 'Total Desc_ Linha', 'Custo', 'N_ Detalhes', 'invoice_count_current_purchase', 'customer_tenure_days', 'prior_purchase_count', 'prior_total_spend', 'purchase_count_30d', 'spend_30d', 'purchase_count_90d', 'spend_90d', 'purchase_count_180d', 'spend_180d', 'purchase_count_365d', 'commercial_zone', 'document_series', 'purchase_date', 'random_noise']


In [32]:
Quantos valores em falta existem por linha, nas colunas cols_65

SyntaxError: invalid syntax (4231522238.py, line 1)

In [33]:
df[cols_65].isnull().sum(axis=1).value_counts().sort_index()

0    6469
Name: count, dtype: int64

In [34]:
Quantas linhas têm todas essas colunas em falta

SyntaxError: invalid syntax (3845864600.py, line 1)

In [35]:
df[cols_65].isnull().all(axis=1).sum()

np.int64(0)

In [36]:
Ver as linhas com todas essas colunas em falta 

SyntaxError: invalid syntax (3408797234.py, line 1)

In [37]:
print(df.loc[df[cols_65].isnull().all(axis=1)])

Empty DataFrame
Columns: [Total a Pagar, Total Bruto, Total Liquido, Total Imp_, Total Portes, Total Desc_ Global, Total Desc_ Linha, Custo, N_ Detalhes, invoice_count_current_purchase, repeat_purchase_90d, days_since_previous_purchase, customer_tenure_days, prior_purchase_count, prior_total_spend, prior_mean_purchase_value, prior_std_purchase_value, prior_min_purchase_value, prior_max_purchase_value, prior_mean_gap_days, prior_std_gap_days, purchase_count_30d, spend_30d, purchase_count_90d, spend_90d, purchase_count_180d, spend_180d, purchase_count_365d, spend_365d, customer_id, payment_type, salesperson, commercial_zone, transport_method, warehouse, document_series, purchase_date, random_noise]
Index: []


In [ ]:
Remover essas linhas

In [39]:
df = df.loc[~df[cols_65].isnull().all(axis=1)].copy()

In [ ]:
Confirmar

In [40]:
print("Linhas restantes:", len(df))

Linhas restantes: 6469


In [41]:
O que bate certo, 6534 - 65 = 6469

SyntaxError: invalid syntax (2196936723.py, line 1)

In [42]:
# 1. Confirmar o estado atual
print("Linhas no df:", len(df))
print("Duplicados nas colunas:", df.duplicated().sum())
print("IDs duplicados no índice:", df.index.duplicated().sum())

# 2. Fazer o merge com os clientes
df = df.merge(
    clientes,
    on="customer_id",
    how="left"
)

# 3. Confirmar que o número de linhas não mudou
print("Linhas depois do merge:", len(df))
print("Colunas depois do merge:", df.shape[1])

display(df.head())

Linhas no df: 6469
Duplicados nas colunas: 0
IDs duplicados no índice: 0
Linhas depois do merge: 6469
Colunas depois do merge: 50


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
0,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,1,79.0,126.0,2.0,858.30,429.150000,112.656252,349.49,508.81,47.000000,NaN,0.0,0.00,1.0,349.49,2.0,858.30,2.0,858.30,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,-0.660448,Country CT10: Land of The Final Version,Region RG25: We Will See Later,NaN,Municipality MU89: Council of Last Minute Edits,City CY286: Cached Result Falls,Postal area PA228: Hidden Sheet Sector,Postal code PC444: #REF! Delivery Route,Street ST303: Absolute Reference Terrace,Address AD761: Cell Without a Home,NaN,Excel Is Not A Database Ltd 1013,2024
1,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,1,244.0,616.0,5.0,6722.83,1344.566000,1683.795178,84.38,4222.15,93.000000,117.657696,0.0,0.00,0.0,0.00,0.0,0.00,4.0,2500.68,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,0.924510,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS8: The Pivot Archipelago,Municipality MU11: Council of Last Minute Edits,City CY147: VLOOKUP Bay,Postal area PA260: Hidden Sheet Sector,Postal code PC636: #REF! Delivery Route,NaN,Address AD382: Cell Without a Home,NaN,Excel Is Not A Database Ltd 35,2015
2,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,1,9.0,610.0,90.0,17590.08,195.445333,190.389932,4.54,1002.15,6.752809,6.384073,2.0,111.27,9.0,1928.55,28.0,5290.25,49.0,7899.80,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,-0.258730,Country CT10: Land of The Final Version,Region RG23: We Will See Later,NaN,Municipality MU70: Council of Last Minute Edits,City CY265: Where-less Query Falls,Postal area PA84: Hidden Sheet Sector,Postal code PC200: #REF! Delivery Route,NaN,Address AD723: Cell Without a Home,NaN,Excel Is Not A Database Ltd 826,2017
3,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,0,97.0,97.0,1.0,270.33,270.330000,NaN,270.33,270.33,NaN,NaN,0.0,0.00,0.0,0.00,1.0,270.33,1.0,270.33,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,1.448402,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS7: The Pivot Archipelago,Municipality MU12: Council of Last Minute Edits,NaN,Postal area PA253: Hidden Sheet Sector,Postal code PC558: #REF! Delivery Route,Street ST144: Last Save Court,Address AD399: Cell Without a Home,NaN,Excel Is Not A Database Ltd 530,2018
4,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,1,85.0,1575.0,25.0,12803.02,512.120800,406.696963,22.58,1736.74,62.083333,66.246520,0.0,0.00,2.0,184.13,4.0,2062.50,4.0,2062.50,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,-1.058890,Country CT10: Land of The Final Version,Region RG31: We Will See Later,NaN,Municipality MU142: Council of Last Minute Edits,City CY359: Spill Error Ridge,Postal area PA1

In [43]:
# Colunas acrescentadas pelo merge
novas_colunas = [
    coluna for coluna in clientes.columns
    if coluna != "customer_id"
]

# Resumo das novas variáveis
resumo_clientes = pd.DataFrame({
    "dtype": df[novas_colunas].dtypes.astype(str),
    "nunique": df[novas_colunas].nunique(dropna=True),
    "missing": df[novas_colunas].isna().sum(),
    "missing_pct": (df[novas_colunas].isna().mean() * 100).round(2)
})

display(resumo_clientes.sort_values("missing", ascending=False))

,dtype,nunique,missing,missing_pct
customer_type,float64,0,6469,100.00
street,str,264,3808,58.87
island,str,10,2189,33.84
city,str,259,2128,32.90
postal_area,str,188,278,4.30
postal_code,str,516,223,3.45
municipality,str,150,217,3.35
commercial_region,str,32,35,0.54
country_or_region,str,7,0,0.00
address,str,597,0,0.00


In [46]:
df[novas_colunas].select_dtypes(include="object").nunique(dropna=True).sort_values(ascending=False)

C:\Users\Utilizador\AppData\Local\Temp\ipykernel_8048\3085177749.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df[novas_colunas].select_dtypes(include="object").nunique(dropna=True).sort_values(ascending=False)


company_name         606
address              597
postal_code          516
street               264
city                 259
postal_area          188
municipality         150
commercial_region     32
island                10
country_or_region      7
dtype: int64

In [ ]:
Excluir provavelmente: company_name, address, street e postal_code. Têm demasiadas categorias e podem levar o modelo a memorizar clientes ou locais específicos.

Avaliar com cuidado: city, postal_area e municipality. Podem ter alguma informação geográfica, mas exigem tratamento de categorias raras.

Candidatas mais promissoras: commercial_region, island e country_or_region, porque têm cardinalidade mais baixa.

Avaliar separadamente: customer_since_year, que é numérica e pode representar antiguidade do cliente.

In [47]:
for coluna in ["country_or_region", "commercial_region", "customer_since_year"]:
    print("\n", coluna)
    display(df[coluna].value_counts(dropna=False).head(10))


 country_or_region


country_or_region
Country CT10: Land of The Final Version    6256
Country CT5: Land of The Final Version      194
Country CT7: Land of The Final Version        8
Country CT6: Land of The Final Version        8
Country CT9: Land of The Final Version        1
Country CT8: Land of The Final Version        1
Country CT4: Land of The Final Version        1
Name: count, dtype: int64


 commercial_region


commercial_region
Region RG17: We Will See Later    4069
Region RG23: We Will See Later     560
Region RG18: We Will See Later     364
Region RG24: We Will See Later     232
Region RG29: We Will See Later     211
Region RG27: We Will See Later     192
Region RG31: We Will See Later     163
Region RG28: We Will See Later     119
Region RG9: We Will See Later      119
Region RG19: We Will See Later      75
Name: count, dtype: int64


 customer_since_year


customer_since_year
2015    3532
2017     880
2016     600
2018     462
2019     275
2020     272
2021     207
2023     108
2022     107
2024      26
Name: count, dtype: int64